# Загрузим библиотеки

In [1]:
import os
import shutil
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

try:
    from tqdm import tqdm
except ImportError:
    def tqdm(iterable, *args, **kwargs):
        return iterable

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split, GridSearchCV

from imblearn.over_sampling import SMOTE

import xgboost as xgb
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold

# Загрузка

In [25]:
# Данные разбиты на 12 файлов-партиций (parquet). 
# Мы читаем их последовательно, чтобы не перегружать память.
# Для каждого файла строим агрегированные признаки на уровне клиента (id):
# - количество кредитных продуктов (rn_count)
# - доли флагов закрытия (pclose_flag, fclose_flag) как частоты
# - one-hot-кодирование бинаризованных и закодированных признаков,
#   при этом редкие категории (встречаемость < 250) отбрасываем для уменьшения шума.
# Затем объединяем все агрегаты в один датафрейм.

In [2]:
PATH = 'train_data/'

In [43]:
import os
import pandas as pd
import tqdm


def read_parquet_dataset_from_local(path_to_dataset: str, start_from: int = 0,
                                     num_parts_to_read: int = 2, columns=None, verbose=False) -> pd.DataFrame:
    """
    читает num_parts_to_read партиций, преобразовывает их к pd.DataFrame и возвращает
    :param path_to_dataset: путь до директории с партициями
    :param start_from: номер партиции, с которой нужно начать чтение
    :param num_parts_to_read: количество партиций, которые требуется прочитать
    :param columns: список колонок, которые нужно прочитать из партиции
    :return: pd.DataFrame
    """

    res = []
    dataset_paths = sorted([os.path.join(path_to_dataset, filename) for filename in os.listdir(path_to_dataset)
                              if filename.startswith('train')])
    print(dataset_paths)

    start_from = max(0, start_from)
    chunks = dataset_paths[start_from: start_from + num_parts_to_read]
    if verbose:
        print('Reading chunks:\n')
        for chunk in chunks:
            print(chunk)
    for chunk_path in tqdm.notebook.tqdm(chunks, desc="Reading dataset with pandas"):
        print('chunk_path', chunk_path)
        chunk = pd.read_parquet(chunk_path,columns=columns)
        res.append(chunk)

    return pd.concat(res).reset_index(drop=True)

In [1]:
import os
import pandas as pd
import numpy as np
from collections import Counter
import shutil

def prepare_transactions_dataset(path_to_dataset: str,
                                 num_parts_total: int = 12,
                                 save_to_path: str = None,
                                 verbose: bool = True) -> pd.DataFrame:
    from tqdm import tqdm

    print(f"Содержимое папки {path_to_dataset}: {os.listdir(path_to_dataset)}")
    all_files = sorted([
        os.path.join(path_to_dataset, f)
        for f in os.listdir(path_to_dataset)
        if f.startswith('train') and (f.endswith('.pq') or f.endswith('.parquet'))
    ])[:num_parts_total]
    print(f"Найдено файлов train_*: {len(all_files)}")
    if len(all_files) == 0:
        raise ValueError(f"Не найдено ни одного файла train_* с расширением .pq или .parquet в {path_to_dataset}")
    # Импортируем tqdm внутри функции, чтобы избежать конфликтов
    from tqdm import tqdm

    all_files = sorted([
        os.path.join(path_to_dataset, f)
        for f in os.listdir(path_to_dataset)
        if f.startswith('train') and f.endswith('.pq')
    ])[:num_parts_total]

    print(f"Найдено файлов: {len(all_files)}")
    if len(all_files) == 0:
        raise ValueError(f"Не найдено ни одного файла train_*.pq в {path_to_dataset}")

    binary_cols = [
        'pre_since_opened', 'pre_since_confirmed', 'pre_pterm', 'pre_fterm',
        'pre_till_pclose', 'pre_till_fclose', 'pre_loans_credit_limit',
        'pre_loans_next_pay_summ', 'pre_loans_outstanding', 'pre_loans_total_overdue',
        'pre_loans_max_overdue_sum', 'pre_loans_credit_cost_rate', 'pre_loans5',
        'pre_loans530', 'pre_loans3060', 'pre_loans6090', 'pre_loans90',
        'pre_util', 'pre_over2limit', 'pre_maxover2limit'
    ]

    coded_cols = [
        'enc_paym_0', 'enc_paym_1', 'enc_paym_2', 'enc_paym_3', 'enc_paym_4',
        'enc_paym_5', 'enc_paym_6', 'enc_paym_7', 'enc_paym_8', 'enc_paym_9',
        'enc_paym_10', 'enc_paym_11', 'enc_paym_12', 'enc_paym_13', 'enc_paym_14',
        'enc_paym_15', 'enc_paym_16', 'enc_paym_17', 'enc_paym_18', 'enc_paym_19',
        'enc_paym_20', 'enc_paym_21', 'enc_paym_22', 'enc_paym_23', 'enc_paym_24',
        'enc_loans_account_holder_type', 'enc_loans_credit_status',
        'enc_loans_credit_type', 'enc_loans_account_cur'
    ]

    flag_cols = [
        'is_zero_loans5', 'is_zero_loans530', 'is_zero_loans3060',
        'is_zero_loans6090', 'is_zero_loans90', 'is_zero_util',
        'is_zero_over2limit', 'is_zero_maxover2limit', 'pclose_flag', 'fclose_flag'
    ]

    ohe_cols = binary_cols + coded_cols

    if verbose:
        print("Первый проход: сбор статистики категорий...")

    counters = {col: Counter() for col in ohe_cols}
    for file_path in tqdm(all_files, desc="Сбор частот"):
        chunk = pd.read_parquet(file_path, columns=ohe_cols)
        if verbose:
            print(f"  Прочитан файл {os.path.basename(file_path)}, строк: {len(chunk)}")
        for col in ohe_cols:
            counters[col].update(chunk[col].dropna().values)

    keep_categories = {}
    for col in ohe_cols:
        keep = [val for val, cnt in counters[col].items() if cnt >= 250]
        keep_categories[col] = keep
        if verbose:
            print(f"  {col}: оставлено {len(keep)} категорий из {len(counters[col])}")

    del counters  # освобождаем память

    if verbose:
        print("Второй проход: агрегация по файлам...")

    temp_dir = os.path.join(path_to_dataset, 'temp_agg')
    os.makedirs(temp_dir, exist_ok=True)

    for idx, file_path in enumerate(tqdm(all_files, desc="Обработка файлов")):
        cols_to_read = ['id', 'rn'] + flag_cols + ohe_cols
        chunk = pd.read_parquet(file_path, columns=cols_to_read)
        if verbose:
            print(f"  Обрабатывается {os.path.basename(file_path)}, строк: {len(chunk)}")

        ohe_list = []
        for col in ohe_cols:
            keep = keep_categories[col]
            if not keep:
                continue
            for val in keep:
                col_name = f"{col}_{val}"
                ohe_list.append((col_name, (chunk[col] == val).astype('int8')))
        ohe_df = pd.DataFrame(dict(ohe_list), index=chunk.index)

        rn_agg = chunk.groupby('id')['rn'].count().rename('rn_count')
        flag_agg = chunk.groupby('id')[flag_cols].sum()
        ohe_agg = ohe_df.groupby(chunk['id']).sum()

        agg_file = pd.concat([rn_agg, flag_agg, ohe_agg], axis=1).reset_index()
        out_path = os.path.join(temp_dir, f"agg_chunk_{idx:02d}.parquet")
        agg_file.to_parquet(out_path, index=False)
        if verbose:
            print(f"    Сохранён агрегат для {len(agg_file)} клиентов")

        # Освобождаем память
        del chunk, ohe_df, rn_agg, flag_agg, ohe_agg, agg_file

    if verbose:
        print("Объединение агрегированных блоков...")

    agg_files = sorted([os.path.join(temp_dir, f) for f in os.listdir(temp_dir) if f.startswith('agg_chunk')])
    print(f"Найдено агрегированных файлов: {len(agg_files)}")

    if len(agg_files) == 0:
        print("Содержимое временной папки:", os.listdir(temp_dir))
        raise RuntimeError("Не создано ни одного агрегированного файла. Проверьте, что данные читаются корректно.")

    df_agg = pd.concat([pd.read_parquet(f) for f in agg_files], ignore_index=True)

    # Удаляем временную папку
    shutil.rmtree(temp_dir)

    # Удаляем is_zero_*
    cols_to_drop = [col for col in df_agg.columns if col.startswith('is_zero_')]
    df_agg.drop(columns=cols_to_drop, inplace=True)

    cols_till_pclose = [col for col in df_agg.columns if col.startswith('pre_till_pclose')]
    df_agg.drop(columns=cols_till_pclose, inplace=True)

    cols_pterm = [col for col in df_agg.columns if col.startswith('pre_pterm')]
    df_agg.drop(columns=cols_pterm, inplace=True)

    for flag in flag_cols:
        if flag in df_agg.columns:
            df_agg[flag] = df_agg[flag] / df_agg['rn_count']

    months = list(range(25))
    codes = [0, 1, 2, 3, 4]

    for month in months:
        cols_month = [f'enc_paym_{month}_{code}' for code in codes if f'enc_paym_{month}_{code}' in df_agg.columns]
        if cols_month:
            df_agg[f'paym_total_{month}'] = df_agg[cols_month].sum(axis=1)

    for month in months:
        cols_month = [f'enc_paym_{month}_{code}' for code in codes if f'enc_paym_{month}_{code}' in df_agg.columns]
        if cols_month:
            weighted_sum = sum(
                code * df_agg[f'enc_paym_{month}_{code}']
                for code in codes
                if f'enc_paym_{month}_{code}' in df_agg.columns
            )
            df_agg[f'paym_avg_{month}'] = weighted_sum / df_agg[f'paym_total_{month}'].replace(0, np.nan)

    for month in months:
        cols_overdue = [f'enc_paym_{month}_{code}' for code in codes if code > 1 and f'enc_paym_{month}_{code}' in df_agg.columns]
        if cols_overdue:
            df_agg[f'paym_overdue_{month}'] = df_agg[cols_overdue].sum(axis=1) / df_agg[f'paym_total_{month}'].replace(0, np.nan)

    paym_avg_cols = [f'paym_avg_{m}' for m in months if f'paym_avg_{m}' in df_agg.columns]
    df_agg['paym_avg_total'] = df_agg[paym_avg_cols].mean(axis=1)

    paym_overdue_cols = [f'paym_overdue_{m}' for m in months if f'paym_overdue_{m}' in df_agg.columns]
    df_agg['paym_overdue_ratio'] = df_agg[paym_overdue_cols].mean(axis=1)

    last6 = [f'paym_avg_{m}' for m in range(19, 25) if f'paym_avg_{m}' in df_agg.columns]
    first6 = [f'paym_avg_{m}' for m in range(0, 6) if f'paym_avg_{m}' in df_agg.columns]
    if last6 and first6:
        df_agg['paym_trend'] = df_agg[last6].mean(axis=1) - df_agg[first6].mean(axis=1)

    if paym_avg_cols:
        df_agg['paym_volatility'] = df_agg[paym_avg_cols].std(axis=1)

    cols_to_drop_final = [
        col for col in df_agg.columns
        if col.startswith('enc_paym_')
           or col.startswith('paym_total_')
           or (col.startswith('paym_avg_') and col not in ['paym_avg_total'])
           or (col.startswith('paym_overdue_') and col not in ['paym_overdue_ratio'])
    ]
    df_agg.drop(columns=cols_to_drop_final, inplace=True)

    manual_drop = [
        'paym_overdue_ratio',
        'pre_loans_next_pay_summ_2',
        'pre_fterm_8',
        'enc_loans_credit_type_4',
        'enc_loans_credit_type_5',
        'enc_loans_credit_status_2',
        'pre_fterm_1',
        'pre_fterm_3'
    ]
    df_agg.drop(columns=[c for c in manual_drop if c in df_agg.columns], inplace=True, errors='ignore')

    if save_to_path:
        os.makedirs(save_to_path, exist_ok=True)
        out_file = os.path.join(save_to_path, 'processed_full.parquet')
        df_agg.to_parquet(out_file, index=False)
        if verbose:
            print(f"Сохранено в {out_file}")

    if verbose:
        print(f"Итоговое число признаков: {df_agg.shape[1]}")

    return df_agg

In [51]:
data_processed = prepare_transactions_dataset(
    path_to_dataset='train_data/',
    num_parts_total=12,
    save_to_path='traindata/',
    verbose=True
)

Содержимое папки train_data/: ['train_data_0.pq', 'train_data_1.pq', 'train_data_10.pq', 'train_data_11.pq', 'train_data_2.pq', 'train_data_3.pq', 'train_data_4.pq', 'train_data_5.pq', 'train_data_6.pq', 'train_data_7.pq', 'train_data_8.pq', 'train_data_9.pq']
Найдено файлов train_*: 12
Найдено файлов: 12
Первый проход: сбор статистики категорий...


Сбор частот:   0%|                                                                              | 0/12 [00:00<?, ?it/s]

  Прочитан файл train_data_0.pq, строк: 1974724


Сбор частот:   8%|█████▊                                                                | 1/12 [00:15<02:49, 15.41s/it]

  Прочитан файл train_data_1.pq, строк: 2107305


...

Обработка файлов:  92%|██████████████████████████████████████████████████████████▋     | 11/12 [09:37<00:50, 50.43s/it]

    Сохранён агрегат для 250000 клиентов
  Обрабатывается train_data_9.pq, строк: 2284256


Обработка файлов: 100%|████████████████████████████████████████████████████████████████| 12/12 [11:28<00:00, 57.40s/it]

    Сохранён агрегат для 250000 клиентов
Объединение агрегированных блоков...
Найдено агрегированных файлов: 12


Сохранено в traindata/processed_full.parquet
Итоговое число признаков: 233


In [52]:
# Загружаем целевую переменную
targets = pd.read_csv('train_target.csv')

# Объединяем с нашими агрегированными данными
full_data = data_processed.merge(targets, on='id', how='inner')
print(f"Размер после слияния: {full_data.shape}")
full_data.head()

# Сохраняем полный датасет для дальнейшего использования
full_data.to_parquet('full_dataset_with_target.parquet')

Размер после слияния: (3000000, 234)


In [53]:
targets.shape

(3000000, 2)

In [54]:
full_data.to_csv('full_dataset.csv')

# Анализ full_data

In [56]:
full_data.head()

,id,rn_count,pclose_flag,fclose_flag,pre_since_opened_18,pre_since_opened_4,pre_since_opened_5,pre_since_opened_3,pre_since_opened_2,pre_since_opened_1,...,enc_loans_credit_type_2,enc_loans_credit_type_7,enc_loans_credit_type_6,enc_loans_account_cur_1,enc_loans_account_cur_2,enc_loans_account_cur_0,paym_avg_total,paym_trend,paym_volatility,flag
0,0,10,0.100000,0.200000,3,1,2,1,1,1,...,0,0,0,10,0,0,1.708000,2.416667,0.976780,0
1,1,14,0.071429,0.142857,1,0,0,0,1,0,...,0,0,0,14,0,0,1.771429,1.869048,0.772343,0
2,2,3,0.666667,0.666667,0,0,0,0,0,0,...,0,0,0,3,0,0,1.986667,1.777778,0.920346,0
3,3,15,0.333333,0.400000,1,2,1,0,1,3,...,0,0,0,15,0,0,1.152000,1.833333,0.825187,0
4,4,1,1.000000,1.000000,0,0,0,0,0,0,...,0,0,0,1,0,0,3.120000,0.333333,0.331662,0


In [ ]:
full_data.shape

In [2]:
full_data = pd.read_csv('full_dataset.csv')

In [7]:
extra_drop = [
    'pre_over2limit_2',
    'pre_loans_total_overdue_0',
    'pre_loans_outstanding_3',
    'pre_util_9',
    'pre_loans3060_5',
    'pre_loans5_6',
    'pre_loans90_8',
    'enc_loans_credit_status_3',
    'pre_loans530_16',
    'enc_loans_account_holder_type_1',
    'pre_loans6090_4',
    'enc_loans_account_cur_1',
    'pre_loans_max_overdue_sum_2',
    'pre_maxover2limit_4',
    'pre_maxover2limit_17'
]

In [10]:
full_data = full_data.drop(columns = extra_drop)

In [11]:
full_data.to_csv('full_dataset_new.csv')

In [12]:
full_data.shape

(3000000, 220)

# Обучение

In [13]:
# Выборка несбалансирована, поэтому требуется предподготовка данных.

In [ ]:
full_data = pd.read_csv('full_dataset_new.csv')

In [ ]:
full_data = full_data.drop('id', axis = 1)

In [ ]:
y = full_data['flag']

In [ ]:
X = full_data.drop('flag', axis = 1)

In [6]:
(y.value_counts()/len(y))*100

flag
0    96.451933
1     3.548067
Name: count, dtype: float64

### Почему отказались от SMOTE и какие модели выбраны

В ходе работы над проектом выяснилось, что целевой класс (дефолт) составляет всего около 3.5% от всех записей - классический случай сильного дисбаланса. Изначально рассматривался вариант с SMOTE, но от него пришлось отказаться по нескольким причинам:

- При 3 миллионах строк синтетическая генерация может увеличить датасет в 2–3 раза, что чревато переполнением памяти и многократным ростом времени обучения.
- SMOTE строит новые объекты на основе ближайших соседей, а в данных много категориальных признаков, полученных через One-Hot Encoding, - это может внести дополнительный шум и ухудшить качество модели на реальных данных.

Поэтому было решено использовать **встроенные механизмы балансировки**, которые работают через пересчёт весов классов в функции потерь:

- Для логистической регрессии и случайного леса применяется параметр `class_weight='balanced'`, автоматически выставляющий веса обратно пропорционально частоте классов.
- Для градиентного бустинга (XGBoost, LightGBM) используются аналогичные параметры: `scale_pos_weight` или `class_weight`.

Такой подход не требует дополнительной памяти, не искажает исходное распределение и позволяет модели учиться на всех реальных примерах, что особенно важно при большом объёме данных.

**Какие модели были протестированы:**

- **Логистическая регрессия** – простой и быстрый бейзлайн, дающий представление о линейной разделимости данных.
- **Случайный лес** – устойчив к переобучению, хорошо масштабируется на большое число признаков и позволяет оценить их важность.
- **Градиентный бустинг (LightGBM и XGBoost)** – показывает наилучшие результаты на табличных данных, эффективно обрабатывает нелинейные зависимости.



In [6]:
random = 37

In [7]:
# Разбиение 70/30 с фиксированным random_state для воспроизводимости

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=random, stratify=y)

In [22]:
print("Размеры обучающей выборки:")
print(f"  X_train: {X_train.shape}, y_train: {y_train.shape}")
print("Размеры тестовой выборки:")
print(f"  X_test: {X_test.shape}, y_test: {y_test.shape}")

Размеры обучающей выборки:
  X_train: (2100000, 219), y_train: (2100000,)
Размеры тестовой выборки:
  X_test: (900000, 219), y_test: (900000,)


In [12]:
print("\nДоля дефолтов (класс 1):")
print(f"  в train: {y_train.mean():.4f} ({y_train.sum()} записей)")
print(f"  в test:  {y_test.mean():.4f} ({y_test.sum()} записей)")


Доля дефолтов (класс 1):
  в train: 0.0355 (74509 записей)
  в test:  0.0355 (31933 записей)


In [13]:
# проведем базовое обучение на разных моделях, выявим на метрике roc_auc лучшуу и с помощью метода кросс подбора оптимизируем

In [37]:


models = {
    'LogReg': LogisticRegression(class_weight='balanced', max_iter=2000, solver='liblinear', random_state=random),
    'DecisionTree': DecisionTreeClassifier(class_weight='balanced', random_state=random),
    'RandomForest': RandomForestClassifier(class_weight='balanced', n_estimators=100, n_jobs=-1, random_state=random),
    'LightGBM': LGBMClassifier(class_weight='balanced', n_estimators=100, random_state=random, n_jobs=-1),
    'XGBoost': XGBClassifier(n_estimators=100, random_state=random, n_jobs=-1, use_label_encoder=False, eval_metric='logloss'),
    'CatBoost': CatBoostClassifier(iterations=100, depth=6, learning_rate=0.1, auto_class_weights='Balanced', random_seed=random, verbose=False, thread_count=-1)
}

В ходе работы была вывляена нехватка памяти, поэтому было принято решение для экспериментов взять 30% от тренировочных данных, провести эксперимент на них, затем полную модель обучать уже на полных данных, оптимизировать тоже на них

In [9]:
X_train_sample, _, y_train_sample, _ = train_test_split(
    X_train, y_train,
    train_size=0.4,
    random_state=random,
    stratify=y_train
)

In [10]:
# в ходе обучения выявлена ошибка спец символов столбцах таблицы, принято решение о чистке.

In [19]:
def clean_column_names(df):
    # Заменяем всё, что не буква, не цифра и не подчёркивание, на подчёркивание
    df.columns = df.columns.str.replace(r'[^A-Za-z0-9_]+', '_', regex=True)
    # Убираем лишние подчёркивания в начале и конце (опционально)
    df.columns = df.columns.str.strip('_')
    return df

In [12]:
X_train_sample = clean_column_names(X_train_sample)
X_test = clean_column_names(X_test)

In [38]:
for name, model in models.items():
    model.fit(X_train_sample, y_train_sample)
    y_pred = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_pred)
    results[name] = auc
    print(f"{name}: {auc:.4f}")

LogReg: 0.7153
DecisionTree: 0.5173
RandomForest: 0.6822
LightGBM: 0.7373


C:\Users\elvir\anaconda3\Lib\site-packages\xgboost\training.py:200: UserWarning: [08:59:55] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:794: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBoost: 0.7256
CatBoost: 0.7285


In [21]:
# Лучше всего на обучающей выборке себя показал LightGBM, принято решение обучать на нём, при этом использую оптимизацию

In [17]:
param_grid = {
    'n_estimators': [100, 125, 150, 175, 200],          # число деревьев
}


In [18]:
model = LGBMClassifier(class_weight='balanced', random_state=random, verbosity=-1)

In [19]:
gs = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=3,
    scoring='roc_auc',
    n_jobs=1,         
    verbose=1
)

In [20]:
gs.fit(X_train_sample, y_train_sample)

Fitting 3 folds for each of 5 candidates, totalling 15 fits


GridSearchCV(cv=3,
             estimator=LGBMClassifier(class_weight='balanced', random_state=23,
                                      verbosity=-1),
             n_jobs=1, param_grid={'n_estimators': [100, 125, 150, 175, 200]},
             scoring='roc_auc', verbose=1)

In [21]:
print("Лучшие параметры:", gs.best_params_)
print("Лучший ROC AUC на кросс-валидации:", gs.best_score_)

Лучшие параметры: {'n_estimators': 100}
Лучший ROC AUC на кросс-валидации: 0.7381230047664294


In [74]:
param_grid = {
    'max_depth': [5, 10, 15, 20, 25],
    'num_leaves': [31, 63, 127]
}


In [75]:
model = LGBMClassifier(n_estimators = 125, class_weight='balanced', random_state=random, verbosity=-1)

In [76]:
gs = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=3,
    scoring='roc_auc',
    n_jobs=1,         
    verbose=1
)

In [77]:
gs.fit(X_train_sample, y_train_sample)

Fitting 3 folds for each of 15 candidates, totalling 45 fits


GridSearchCV(cv=3,
             estimator=LGBMClassifier(class_weight='balanced', n_estimators=125,
                                      random_state=23, verbosity=-1),
             n_jobs=1,
             param_grid={'max_depth': [5, 10, 15, 20, 25],
                         'num_leaves': [31, 63, 127]},
             scoring='roc_auc', verbose=1)

In [79]:
print("Лучшие параметры:", gs.best_params_)
print("Лучший ROC AUC на кросс-валидации:", gs.best_score_)

Лучшие параметры: {'max_depth': 20, 'num_leaves': 31}
Лучший ROC AUC на кросс-валидации: 0.738587505694745


In [80]:
param_grid = {
    'learning_rate': [0.005, 0.01, 0.05, 0.1, 0.075]
}

In [81]:
model = LGBMClassifier(n_estimators = 125, max_depth = 20, num_leaves = 31, class_weight='balanced', random_state=random, verbosity=-1)

In [82]:
gs = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=3,
    scoring='roc_auc',
    n_jobs=1,         
    verbose=1
)

In [83]:
gs.fit(X_train_sample, y_train_sample)

Fitting 3 folds for each of 5 candidates, totalling 15 fits


GridSearchCV(cv=3,
             estimator=LGBMClassifier(class_weight='balanced', max_depth=20,
                                      n_estimators=125, random_state=23,
                                      verbosity=-1),
             n_jobs=1,
             param_grid={'learning_rate': [0.005, 0.01, 0.05, 0.1, 0.075]},
             scoring='roc_auc', verbose=1)

In [84]:
print("Лучшие параметры:", gs.best_params_)
print("Лучший ROC AUC на кросс-валидации:", gs.best_score_)

Лучшие параметры: {'learning_rate': 0.075}
Лучший ROC AUC на кросс-валидации: 0.7390549749555496


In [103]:
param_grid = {
    'subsample': [0.8, 0.9, 1.0]
}

In [95]:
model = LGBMClassifier(n_estimators = 125, max_depth = 20, num_leaves = 31, learning_rate = 0.075, class_weight='balanced', random_state=random, verbosity=-1)

In [96]:
gs = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=3,
    scoring='roc_auc',
    n_jobs=1,         
    verbose=1
)

In [104]:
gs.fit(X_train_sample, y_train_sample)

Fitting 3 folds for each of 3 candidates, totalling 9 fits


GridSearchCV(cv=3,
             estimator=LGBMClassifier(class_weight='balanced',
                                      learning_rate=0.075, max_depth=20,
                                      n_estimators=125, random_state=23,
                                      subsample=0.8, verbosity=-1),
             n_jobs=1, param_grid={'colsample_bytree': [0.8, 0.9, 0.1]},
             scoring='roc_auc', verbose=1)

In [98]:
print("Лучшие параметры:", gs.best_params_)
print("Лучший ROC AUC на кросс-валидации:", gs.best_score_)

Лучшие параметры: {'subsample': 0.8}
Лучший ROC AUC на кросс-валидации: 0.7390549749555496


In [99]:
param_grid = {
    'colsample_bytree': [0.8, 0.9, 0.1]
}

In [100]:
model = LGBMClassifier(n_estimators = 125, max_depth = 20, num_leaves = 31, learning_rate = 0.075, subsample = 0.8, class_weight='balanced', random_state=random, verbosity=-1)

In [101]:
gs = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=3,
    scoring='roc_auc',
    n_jobs=1,         
    verbose=1
)

In [102]:
gs.fit(X_train_sample, y_train_sample)

Fitting 3 folds for each of 3 candidates, totalling 9 fits


GridSearchCV(cv=3,
             estimator=LGBMClassifier(class_weight='balanced',
                                      learning_rate=0.075, max_depth=20,
                                      n_estimators=125, random_state=23,
                                      subsample=0.8, verbosity=-1),
             n_jobs=1, param_grid={'colsample_bytree': [0.8, 0.9, 0.1]},
             scoring='roc_auc', verbose=1)

In [105]:
print("Лучшие параметры:", gs.best_params_)
print("Лучший ROC AUC на кросс-валидации:", gs.best_score_)

Лучшие параметры: {'colsample_bytree': 0.9}
Лучший ROC AUC на кросс-валидации: 0.7389844008936842


In [20]:
X_train = clean_column_names(X_train)

In [62]:
model = LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.075,
    max_depth=20,
    num_leaves=40,
    subsample=0.8,
    colsample_bytree=0.9,
    class_weight='balanced',
    random_state=37,
    verbosity=-1,
    min_data_in_leaf = 500,
    reg_alpha = 0.0, 
    reg_lambda = 1.0
)

In [63]:
model.fit(X_train, y_train)

LGBMClassifier(class_weight='balanced', colsample_bytree=0.9,
               learning_rate=0.075, max_depth=20, min_data_in_leaf=500,
               n_estimators=1000, num_leaves=40, random_state=37,
               reg_lambda=1.0, subsample=0.8, verbosity=-1)

In [64]:
print(roc_auc_score(y_test, model.predict_proba(X_test)[:, 1]))

0.748051923832185


In [65]:
print(roc_auc_score(y_train, model.predict_proba(X_train)[:, 1]))

0.8270567959801854


In [18]:
param_grid = {
    'min_data_in_leaf': [50, 100, 200, 500]
}

In [19]:
gs = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=3,
    scoring='roc_auc',
    n_jobs=1,         
    verbose=1
)

In [20]:
gs.fit(X_train_sample, y_train_sample)

Fitting 3 folds for each of 4 candidates, totalling 12 fits


GridSearchCV(cv=3,
             estimator=LGBMClassifier(class_weight='balanced',
                                      colsample_bytree=0.9, learning_rate=0.075,
                                      max_depth=20, n_estimators=125,
                                      num_leaves=40, random_state=37,
                                      subsample=0.8, verbosity=-1),
             n_jobs=1, param_grid={'min_data_in_leaf': [50, 100, 200, 500]},
             scoring='roc_auc', verbose=1)

In [21]:
print("Лучшие параметры:", gs.best_params_)
print("Лучший ROC AUC на кросс-валидации:", gs.best_score_)

Лучшие параметры: {'min_data_in_leaf': 500}
Лучший ROC AUC на кросс-валидации: 0.7395083580104606


In [38]:
param_grid = {
    'reg_alpha': [0.0, 0.1, 0.5, 1.0],
    'reg_lambda': [0.0, 0.1, 0.5, 1.0]
}

In [39]:
gs = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=3,
    scoring='roc_auc',
    n_jobs=1,         
    verbose=1
)

In [40]:
gs.fit(X_train_sample, y_train_sample)

Fitting 3 folds for each of 16 candidates, totalling 48 fits


GridSearchCV(cv=3,
             estimator=LGBMClassifier(class_weight='balanced',
                                      colsample_bytree=0.9, learning_rate=0.075,
                                      max_depth=20, min_data_in_leaf=500,
                                      n_estimators=125, num_leaves=40,
                                      random_state=37, subsample=0.8,
                                      verbosity=-1),
             n_jobs=1,
             param_grid={'reg_alpha': [0.0, 0.1, 0.5, 1.0],
                         'reg_lambda': [0.0, 0.1, 0.5, 1.0]},
             scoring='roc_auc', verbose=1)

In [41]:
print("Лучшие параметры:", gs.best_params_)
print("Лучший ROC AUC на кросс-валидации:", gs.best_score_)

Лучшие параметры: {'reg_alpha': 0.0, 'reg_lambda': 1.0}
Лучший ROC AUC на кросс-валидации: 0.7398572886509175


In [26]:
# подборка через гридсерч не даёт необходимого качества,а так же модель переобучается, попробуем модель с раннеё остоновкой на большем количестве деревьевю

In [16]:
from lightgbm import early_stopping

# Увеличим валидационную выборку до 10% (~210 тыс. строк) — память должна выдержать
val_sample = X_train.sample(frac=0.1, random_state=37)
y_val_sample = y_train.loc[val_sample.index]

#  Модель с усиленной регуляризацией
model_final = LGBMClassifier(
    n_estimators=2000,
    learning_rate=0.03,
    max_depth=20,
    num_leaves=31,              # уменьшили с 40
    subsample=0.8,
    colsample_bytree=0.9,
    class_weight='balanced',    
    min_data_in_leaf=1000,      # увеличили
    reg_alpha=0.5,              
    reg_lambda=2.0,             
    random_state=37,
    verbosity=-1
)

# Обучаем с ранней остановкой по AUC
model_final.fit(
    X_train, y_train,
    eval_set=[(val_sample, y_val_sample)],
    eval_metric='auc',                    
    callbacks=[early_stopping(stopping_rounds=50)]
)

# Оценка на тесте
test_auc = roc_auc_score(y_test, model_final.predict_proba(X_test)[:, 1])
print(f"Test AUC: {test_auc:.4f}")

C:\Users\elvir\anaconda3\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[2000]	valid_0's auc: 0.804466	valid_0's binary_logloss: 0.555463
Test AUC: 0.7504


In [17]:
final_model = LGBMClassifier(
    n_estimators=2000,
    learning_rate=0.03,
    max_depth=20,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.9,
    class_weight='balanced',
    min_data_in_leaf=1000,
    reg_alpha=0.5,
    reg_lambda=2.0,
    random_state=37,
    verbosity=-1
)

In [18]:
final_model.fit(X_train, y_train)

LGBMClassifier(class_weight='balanced', colsample_bytree=0.9,
               learning_rate=0.03, max_depth=20, min_data_in_leaf=1000,
               n_estimators=2000, random_state=37, reg_alpha=0.5,
               reg_lambda=2.0, subsample=0.8, verbosity=-1)

In [19]:
print(roc_auc_score(y_test, final_model.predict_proba(X_test)[:, 1]))

0.7503693859791054


In [20]:
print(roc_auc_score(y_train, final_model.predict_proba(X_train)[:, 1]))

0.8003502508173425


In [27]:
# у модели наблюдается переобучение, попробуем ужесточить требования

In [21]:
model_reg = LGBMClassifier(
    n_estimators=2000,
    learning_rate=0.03,
    max_depth=20,
    num_leaves=25,
    subsample=0.8,
    colsample_bytree=0.9,
    class_weight='balanced',
    min_data_in_leaf=2000,
    reg_alpha=1.0,
    reg_lambda=3.0,
    random_state=37,
    verbosity=-1
)

# Обучаем с early stopping на 15% валидации
val_sample = X_train.sample(frac=0.15, random_state=37)
y_val_sample = y_train.loc[val_sample.index]

model_reg.fit(
    X_train, y_train,
    eval_set=[(val_sample, y_val_sample)],
    eval_metric='auc',
    callbacks=[early_stopping(stopping_rounds=30)]
)

C:\Users\elvir\anaconda3\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 30 rounds
Did not meet early stopping. Best iteration is:
[2000]	valid_0's auc: 0.790973	valid_0's binary_logloss: 0.563983


LGBMClassifier(class_weight='balanced', colsample_bytree=0.9,
               learning_rate=0.03, max_depth=20, min_data_in_leaf=2000,
               n_estimators=2000, num_leaves=25, random_state=37, reg_alpha=1.0,
               reg_lambda=3.0, subsample=0.8, verbosity=-1)

In [22]:
test_auc_reg = roc_auc_score(y_test, model_reg.predict_proba(X_test)[:, 1])
print(f"Test AUC (model_reg): {test_auc_reg:.4f}")

Test AUC (model_reg): 0.7505


In [23]:
import pickle
with open('model_reg.pkl', 'wb') as f:
    pickle.dump(model_reg, f)

# Пайплайн

In [2]:
# Параметры модели
best_params = {
    'n_estimators': 2000,
    'learning_rate': 0.03,
    'max_depth': 20,
    'num_leaves': 25,
    'subsample': 0.8,
    'colsample_bytree': 0.9,
    'class_weight': 'balanced',
    'min_data_in_leaf': 2000,
    'reg_alpha': 1.0,
    'reg_lambda': 3.0,
    'random_state': 37,
    'verbosity': -1
}

# Создаём пайплайн
pipeline = Pipeline([
    ('variance', VarianceThreshold(threshold=0.0)),   # удаляем признаки с нулевой дисперсией
    ('classifier', LGBMClassifier(**best_params))
])

In [ ]:
pipeline.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import roc_auc_score
probs_test = pipeline.predict_proba(X_test)[:, 1]
auc_test = roc_auc_score(y_test, probs_test)
print(f"ROC-AUC на тесте (через пайплайн): {auc_test:.4f}")